<a href="https://colab.research.google.com/github/shahriariit/ML_Airlines_Services/blob/main/dataset_1_sentiment_analysis_and_model_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install --upgrade pip
!pip install tensorflow
!pip install torch
!pip install transformers
!pip install tqdm
!pip install nltk pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.2 MB/s eta 0:00:00


In [3]:
import re
import numpy as np
import pandas as pd
import nltk
import torch
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import *
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.calibration import CalibratedClassifierCV

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Conv1D, GlobalMaxPooling1D, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import SpatialDropout1D, Dropout, BatchNormalization, MaxPooling1D, Input
from tensorflow.keras.callbacks import EarlyStopping
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import RandomizedSearchCV
import optuna
from sklearn.model_selection import cross_val_score


# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added this line

True

In [4]:
MAX_WORDS = 10000
MAX_LEN   = 100
EMBED_DIM = 100
TEST_SIZE = 0.2
VAL_SPLIT = 0.1
RANDOM_STATE = 42

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = nltk.word_tokenize(text)
    stop_words = set(nltk.corpus.stopwords.words('english'))
    lemmatizer = nltk.stem.WordNetLemmatizer()

    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

In [6]:
def preprocess_dataframe(df):
    df = df.copy()
    df = df.drop(columns=['title', 'airline'], errors='ignore')

    print("Cleaning text...")
    df['cleaned_review'] = df['review'].astype(str).apply(clean_text)

    df = df.drop(columns=['review'], errors='ignore')

    le = LabelEncoder()
    df['target'] = le.fit_transform(df['target'])

    print("Label Encoding:", dict(zip(le.classes_, le.transform(le.classes_))))
    return df, le

In [7]:
def split_data(df):
    X = df['cleaned_review']
    y = df['target']

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)

    return X_train, X_val, X_test, y_train, y_val, y_test

In [8]:
def calculate_metrics(y_true, y_pred, y_proba=None, name="Model"):
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall    = recall_score(y_true, y_pred, average='weighted')
    f1        = f1_score(y_true, y_pred, average='weighted')

    auc = np.nan

    if y_proba is not None and len(np.unique(y_true)) > 1:
        try:
            y_proba = np.array(y_proba)

            if y_proba.ndim == 1:
                y_proba = np.vstack([-y_proba, y_proba]).T
            if y_proba.shape[1] == 2:
                auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                auc = roc_auc_score(
                    y_true,
                    y_proba,
                    multi_class='ovr',
                    average='weighted'
                )

        except Exception:
            auc = np.nan

    print(f"\n{name}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc
    }

In [9]:
def run_classic_ml(X_train, X_test, y_train, y_test):

    vectorizer = TfidfVectorizer(max_features=MAX_WORDS, ngram_range=(1,2))

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec  = vectorizer.transform(X_test)

    models = {
        "LogReg": LogisticRegression(),
        "NaiveBayes": MultinomialNB(),
        "LinearSVM": LinearSVC(),
        "DecisionTree": DecisionTreeClassifier(max_depth=20, random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
        "ExtraTrees": ExtraTreesClassifier( n_estimators=200, random_state=42),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200),
        "AdaBoost": AdaBoostClassifier(n_estimators=200),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "SGD": SGDClassifier(),
        "PassiveAggressive": PassiveAggressiveClassifier()

    }

    results = {}

    for idx, (name, model) in enumerate(models.items(), 1):
        print(f"\n[{idx}] Training {name}...")
        print("-"*60)

        model.fit(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)

        y_proba = None
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test_vec)

        results[name] = calculate_metrics(y_test, y_pred, y_proba, name)

    return results

In [10]:
def prepare_dl_data(X_train, X_val, X_test, y_train, y_val, y_test):

    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN)
    X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN)
    X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN)

    return X_train_seq, X_val_seq, X_test_seq, y_train.values, y_val.values, y_test.values

In [11]:
def create_lstm():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        LSTM(128),
        Dense(3, activation='softmax')
    ])

def create_gru():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        GRU(128),
        Dense(3, activation='softmax')
    ])

def create_cnn():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(3, activation='softmax')
    ])

def create_bilstm():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(128)),
        Dense(3, activation='softmax')
    ])

In [12]:
def train_dl_model(name, model_fn, X_train, y_train, X_test, y_test):

    print(f"\nTraining {name}...")

    model = model_fn()
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=VAL_SPLIT, verbose=0)

    y_proba = model.predict(X_test)
    y_pred  = np.argmax(y_proba, axis=1)

    metrics = calculate_metrics(y_test, y_pred, y_proba, name)
    return metrics

In [13]:
print("Loading dataset...")
df = pd.read_csv("https://raw.githubusercontent.com/shahriariit/opendataset/master/Bangladesh-airline-dataset.csv")

df, le = preprocess_dataframe(df)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

Loading dataset...
Cleaning text...
Label Encoding: {'Mixed': np.int64(0), 'Negative': np.int64(1), 'Positive': np.int64(2)}


In [14]:
df.to_csv('cleaned_dataframe.csv', index=False)

In [15]:
classic_ml_results = run_classic_ml(X_train, X_test, y_train, y_test)


[1] Training LogReg...
------------------------------------------------------------

LogReg
Accuracy : 0.7905
Precision: 0.7906
Recall   : 0.7905
F1 Score : 0.7904
AUC      : 0.9186

[2] Training NaiveBayes...
------------------------------------------------------------

NaiveBayes
Accuracy : 0.7714
Precision: 0.7839
Recall   : 0.7714
F1 Score : 0.7731
AUC      : 0.9185

[3] Training LinearSVM...
------------------------------------------------------------

LinearSVM
Accuracy : 0.8286
Precision: 0.8323
Recall   : 0.8286
F1 Score : 0.8280
AUC      : nan

[4] Training DecisionTree...
------------------------------------------------------------

DecisionTree
Accuracy : 0.5810
Precision: 0.5986
Recall   : 0.5810
F1 Score : 0.5794
AUC      : 0.7295

[5] Training RandomForest...
------------------------------------------------------------

RandomForest
Accuracy : 0.8286
Precision: 0.8377
Recall   : 0.8286
F1 Score : 0.8296
AUC      : 0.9344

[6] Training ExtraTrees...
----------------------

In [16]:
X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl = prepare_dl_data(
    X_train, X_val, X_test, y_train, y_val, y_test
)

dl_models = {
    "LSTM": create_lstm,
    "GRU": create_gru,
    "CNN": create_cnn,
    "BiLSTM": create_bilstm
}
dl_results = {}
for name, fn in dl_models.items():
    metrics = train_dl_model(name, fn, X_train_dl, y_train_dl, X_test_dl, y_test_dl)
    dl_results[name] = metrics


Training LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

LSTM
Accuracy : 0.7524
Precision: 0.7636
Recall   : 0.7524
F1 Score : 0.7542
AUC      : 0.9146

Training GRU...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

GRU
Accuracy : 0.7048
Precision: 0.7509
Recall   : 0.7048
F1 Score : 0.7096
AUC      : 0.9079

Training CNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



CNN
Accuracy : 0.7333
Precision: 0.7371
Recall   : 0.7333
F1 Score : 0.7346
AUC      : 0.8834

Training BiLSTM...
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step

BiLSTM
Accuracy : 0.7810
Precision: 0.8068
Recall   : 0.7810
F1 Score : 0.7821
AUC      : 0.9227


In [17]:
class TorchTextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        # labels: pandas Series or np.array or list
        if hasattr(labels, 'values'):
            labels = labels.values
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.encodings = encodings
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item
    def __len__(self):
        return len(self.labels)

In [18]:
def pytorch_transformer_tokenize(tokenizer, texts, max_length):
    return tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length)

def train_pytorch_transformer(
    model, tokenizer,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    label_encoder,
    max_length=128,
    model_name="Model"
):

    print(f"\n{'='*70}")
    print(f"TRAINING {model_name}")
    print(f"{'='*70}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    #  ensure correct classification setup
    model.config.problem_type = "single_label_classification"

    # Tokenization
    train_encodings = pytorch_transformer_tokenize(tokenizer, X_train, max_length)
    val_encodings   = pytorch_transformer_tokenize(tokenizer, X_val, max_length)
    test_encodings  = pytorch_transformer_tokenize(tokenizer, X_test, max_length)

    train_dataset = TorchTextDataset(train_encodings, y_train)
    val_dataset   = TorchTextDataset(val_encodings, y_val)
    test_dataset  = TorchTextDataset(test_encodings, y_test)

    # TrainingArguments
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.lower()}",
        num_train_epochs=3,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        learning_rate=2e-5,
        warmup_ratio=0.1,

        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",

        max_grad_norm=1.0,

        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset
    )

    # Train
    trainer.train()

    # Predict
    test_outputs = trainer.predict(test_dataset)
    logits = test_outputs.predictions

    # Predictions
    y_pred = np.argmax(logits, axis=1)

    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)
    probs = torch.nan_to_num(probs)
    y_proba = probs.cpu().numpy()

    # Evaluation
    results = calculate_metrics(
        y_test,
        y_pred,
        y_proba,
        model_name
    )

    return results

In [19]:
transformer_models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "ALBERT": "albert-base-v2",
    #"DeBERTa": "microsoft/deberta-v3-base"
}

In [20]:
num_classes = len(le.classes_)
print(f"Number of classes: {num_classes}")

print("\n TRAINING TRANSFORMERS...")
print("="*70)
transformer_results = {}
for name, checkpoint in transformer_models.items():
    print(f"\n Loading {name}...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=num_classes
    )
    metrics = train_pytorch_transformer(
        model,
        tokenizer,
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        le,
        model_name=name
    )
    transformer_results[name] = metrics

Number of classes: 3

 TRAINING TRANSFORMERS...

 Loading BERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



TRAINING ALBERT


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,1.027415,0.892031
2,0.747193,0.600965
3,0.506266,0.554019



ALBERT
Accuracy : 0.6857
Precision: 0.6893
Recall   : 0.6857
F1 Score : 0.6872
AUC      : 0.8728


In [22]:
all_results = {
    **classic_ml_results,
    **dl_results,
    **transformer_results
}

# Convert to DataFrame
results_df = pd.DataFrame.from_dict(all_results, orient='index')
results_df.index.name = 'Model'
print("\n--- Summary of Model Performance ---\n")
print(results_df.round(4))


--- Summary of Model Performance ---

                   Accuracy  Precision  Recall  F1 Score     AUC
Model                                                           
LogReg               0.7905     0.7906  0.7905    0.7904  0.9186
NaiveBayes           0.7714     0.7839  0.7714    0.7731  0.9185
LinearSVM            0.8286     0.8323  0.8286    0.8280     NaN
DecisionTree         0.5810     0.5986  0.5810    0.5794  0.7295
RandomForest         0.8286     0.8377  0.8286    0.8296  0.9344
ExtraTrees           0.8095     0.8259  0.8095    0.8103  0.9455
GradientBoosting     0.7429     0.7516  0.7429    0.7446  0.8831
AdaBoost             0.7143     0.7126  0.7143    0.7128  0.8297
KNN                  0.6762     0.6771  0.6762    0.6760  0.8315
SGD                  0.8286     0.8340  0.8286    0.8276     NaN
PassiveAggressive    0.8286     0.8309  0.8286    0.8273     NaN
LSTM                 0.7524     0.7636  0.7524    0.7542  0.9146
GRU                  0.7048     0.7509  0.7048    0

In [23]:
top3_df = results_df.sort_values(by="AUC", ascending=False).head(3)

print("\n--- Top 3 Models ---\n")
print(top3_df)

top3_names = top3_df.index.tolist()


--- Top 3 Models ---

              Accuracy  Precision    Recall  F1 Score       AUC
Model                                                          
ExtraTrees    0.809524   0.825926  0.809524  0.810256  0.945510
RandomForest  0.828571   0.837653  0.828571  0.829641  0.934354
BiLSTM        0.780952   0.806772  0.780952  0.782143  0.922721
